# Libraries

In [2]:
import pandas as pd
import os

# EDA

In [8]:
dataset_path = os.path.join('..', 'datasets', 'Philippine Fake News Corpus.csv')
df = pd.read_csv(dataset_path)
df.head()

,Headline,Content,Authors,Date,URL,Brand,Label
0,PH ranks 2nd in Asia-Pacific in deaths due to ...,Pollution caused by traditional cooking fuel i...,['Philippine Daily Inquirer'],NaN,https://newsinfo.inquirer.net/987262/ph-ranks-...,Inquirer,Credible
1,"Aguirre, PCSO chief deny plotting to kill ‘Ato...",Justice Secretary Vitaliano Aguirre 2nd and Ph...,['Jomar Canlas'],2017-04-28 20:12:54+00:00,https://www.manilatimes.net/aguirre-pcso-chief...,Manila Times,Credible
2,Duterte says charges vs ex-President will fail,President Rodrigo Duterte on Monday night desc...,['Christine O. Avendaño'],NaN,https://newsinfo.inquirer.net/914727/duterte-s...,Inquirer,Credible
3,Group warns BFAR on law enforcement fund,THE militant fisher folk group Pambansang Laka...,['Neil Alcober'],2017-08-12 19:54:48+00:00,https://www.manilatimes.net/group-warns-bfar-l...,Manila Times,Credible
4,Solon asks Duterte for jet ski to Panatag,Magdalo Rep. Gary Alejano is willing to lead t...,['Dj Yap'],NaN,https://newsinfo.inquirer.net/882744/solon-ask...,Inquirer,Credible


## Central Statistics of the Dataset

In [9]:
df.describe()

,Headline,Content,Authors,Date,URL,Brand,Label
count,22458,22458,22458,13317,22458,22458,22458
unique,22397,22397,1531,6161,22458,13,2
top,REGULAR AND NON-WORKING HOLIDAYS IN 2017,On President Rodrigo Duterte’s 100 days of off...,[],2017-11-06 00:00:00,https://newsinfo.inquirer.net/987262/ph-ranks-...,Inquirer,Credible
freq,2,2,6025,18,1,7503,14802


Not much to take note of since our dataset looks like it only has strings as a datatype for all of its columns, which is natural considering we're working on creating a fake news detection model.

Something to take note of, however, is the column counts and unique values. We don't want to have features that are way too diverse for the model to train on, so we would want to have a distinct feature that can separate fake news from authentic news. This means that any column that has a lot of unique values are columns that we want to avoid to train the model, with the exclusion of the "Content" and "Headline" columns as these are naturally unique.

When it comes to deciding whether we want to keep Headline or Content column, we should definitely opt for the Content column as not only does it contain the overall information of the post, but it also allows the model to learn the words used in the post, allowing it to identify word combinations indicative of fake news against those that are more commonly used in authentic news. Headline, after all, is just the truncated form of the Content column meant to tell the reader what the news is all about, so we'll discard the Headline and keep the Content column to reduce redundancy between the two and still keep the most information for our model to learn from.

We are left with the Authors, Date, URL, and Brand columns. The Authors column, along with the Brand, are both synonymous when it comes to verifying credibility through post author, so we would likely have to choose one or the other, but not both. The URL column is way too unique for the model to generalize a pattern with, so we'll have to likely discard this column as a feature since the model would not learn any meaningful pattern with it. The Date column may prove to be useful when it comes to verifying recency in news content, but the low unique value suggests either many of the rows share the same date or there are missing values in the column, either way, the date column adds little information to news veracity so discarding it would probably be better.

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22458 entries, 0 to 22457
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Headline  22458 non-null  object
 1   Content   22458 non-null  object
 2   Authors   22458 non-null  object
 3   Date      13317 non-null  object
 4   URL       22458 non-null  object
 5   Brand     22458 non-null  object
 6   Label     22458 non-null  object
dtypes: object(7)
memory usage: 1.2+ MB


It looks like we've got some missing values, let's inspect that more.

## Missing Values

In [13]:
df.isna().sum()

Headline       0
Content        0
Authors        0
Date        9141
URL            0
Brand          0
Label          0
dtype: int64

In [22]:
missing_pct = round(
    df.loc[:,'Date'].isna().sum()/len(df) * 100, 
    2
)

print(f'There is about {missing_pct}% of information that would be lost in discarding NaNs.')

There is about 40.7% of information that would be lost in discarding NaNs.


We'll be losing a lot of information if we discard the rows with the Date NaNs; however, we can just omit the Date column in predicting whether or not a given news headline is credible.

With this, we're left with the content, authors, and brand as features for our model to use in predicting.

Before moving onto the data processing, let's also check the dataset distributions.

## Dataset Distributions

### Label Distribution

In [25]:
df.Label.value_counts()

Label
Credible        14802
Not Credible     7656
Name: count, dtype: int64

It looks like our dataset is heavily imbalanced, with the credible label being twice as large as the not credible sources. This may introduce bias for our model as the model would see more credible sources than non-credible ones so we'll have to balance this somehow.

Authors and Brands are redundant in information so we'll have to choose between the two. For this, let's take a look at the distributions for authors and brands.

### Author Distribution

In [31]:
df.Authors.value_counts()

Authors
[]                                                                        6025
['Pol Pinoy']                                                             4012
['Philippine Daily Inquirer']                                             1302
['View All Posts Pinoytrending']                                           940
['The Manila Times']                                                       434
                                                                          ... 
['Erika Sauler', 'Leilanie Adriano', 'Pocholo Concepcion']                   1
['Dj Yap', 'Juan Escandor Jr.', 'Mar S. Arguelles']                          1
['Prison Planet', 'Perfecto Yasay Jr', 'Paul Parenas', 'Franky Breva']       1
['Madonna Virola', 'Maricar Cinco']                                          1
['Germelina Lacorte', 'Jovic Yee']                                           1
Name: count, Length: 1531, dtype: int64

This looks like its a lot, and it looks like these are combinations of multiple authors under one post, which could make prediction more complex. Albeit including this in our model's prediction would probably increase the validity of its prediction, the complexity of the model would drastically increase if we were to model it for all the possible combinations of individuals.

We could also model it to identify certain individuals and associate them with a higher credibility score for the model to learn, but then if the news organization were to introduce new individuals, the model would likely label them as non-credible due to their names not being trained as a credible name.

Including invidivual names may also introduce racial bias for the model, which we clearly do not want to happen.

An alternative would be to focus on the organization name instead, as all individuals that work under a specific news organization is definitely trustworthy when it comes to delivering actual/valid news.

### Brand Distribution

In [28]:
df.Brand.value_counts()

Brand
Inquirer                    7503
Manila Bulletin             4140
Adobo Chronicles            4015
Manila Times                3159
Pinoytrending Altervista     940
GRPundit                     884
Get Real Philippines         621
Duterte Daily Stories        417
Thinking Pinoy               247
Pinoy News Blogger           223
News Media PH                145
Pilipinas Online Updates     120
Hot News Philippines          44
Name: count, dtype: int64

The brand distribution in our dataset seems to be more finite compared to the authors. It also avoids unethical bias from the model while still retaining credibility from post author. We can see some very familiar names here like Manila Times and Inquirer, alongside some very suspicious brand names such as Adobo Chronicles.

This seems like a great feature for our model to start learning from, so we would likely want to include this as a feature.

With this, the columns we will be using as a feature would be the Content and Brand column. Using this, the model can learn about the content and what makes it credible alongside the post author organization through the brand name of the news.

However, before we move on to training the model, we need to first process the data in a way the model can understand.